# 04 - Transcriptomic Analysis with CGCS

**Applying Residue Manifold Learning to Gene Expression in Trisomy 21**  
**Practical version for Allen Lab exploration**

In [ ]:
# ================================================
# SETUP - Run this first
# ================================================
import sys
from pathlib import Path

# Robust path setup
if "/content/allen-lab-report-tool" not in str(Path.cwd()):
    repo_path = "/content/allen-lab-report-tool"
    if Path(repo_path).exists():
        %cd /content/allen-lab-report-tool
    else:
        print("Cloning repo...")
        !git clone https://github.com/thinkthoughts/allen-lab-report-tool.git
        %cd allen-lab-report-tool

sys.path.insert(0, str(Path.cwd() / "src"))

import grok
from grok.trisomy_metrics import trisomy_cgcs_score
from grok.visualization import plot_cgcs_vs_noise

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Setup complete")
print(f"Version: {grok.__version__}")

## 1. Simulate Realistic Transcriptomic Data

In [ ]:
np.random.seed(42)
n_genes = 1000

# Normal (euploid) expression
normal_expr = np.random.normal(loc=8.0, scale=1.8, size=n_genes)

# Trisomy 21 simulation
trisomy_expr = normal_expr.copy()
chr21_genes = np.random.choice(n_genes, 150, replace=False)  # ~15% genes affected
trisomy_expr[chr21_genes] *= 1.5
trisomy_expr += np.random.normal(0, 0.4, n_genes)   # global dysregulation

print(f"Simulated {n_genes} genes ({len(chr21_genes)} chr21-like genes)")

## 2. CGCS Scoring on Transcriptomic Data

In [ ]:
def transcriptomic_cgcs(normal, trisomy):
    fc = trisomy / (normal + 1e-8)
    mean_fc = np.mean(fc)
    imbalance = np.abs(mean_fc - 1.0)
    dysregulation = np.std(fc)
    
    return trisomy_cgcs_score(
        dosage_ratio=mean_fc,
        overexpression_imbalance=imbalance,
        global_dysregulation=dysregulation,
        return_components=True
    )

result = transcriptomic_cgcs(normal_expr, trisomy_expr)

print("Transcriptomic CGCS Results:")
for k, v in result.items():
    print(f"  {k:25} = {v:.4f}")

## 3. Visualizations

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(normal_expr, color='blue', alpha=0.6, kde=True, label='Normal')
sns.histplot(trisomy_expr, color='red', alpha=0.6, kde=True, label='Trisomy 21')
plt.title("Gene Expression Distribution: Normal vs Trisomy 21")
plt.xlabel("Log2 Expression Level")
plt.ylabel("Gene Count")
plt.legend()
plt.show()

## 4. CGCS vs Overexpression Strength

In [ ]:
over_levels = np.linspace(1.0, 1.8, 12)
cgcs_scores = []

for level in over_levels:
    temp = normal_expr.copy()
    temp[chr21_genes] = normal_expr[chr21_genes] * level
    score = transcriptomic_cgcs(normal_expr, temp)['cgcs']
    cgcs_scores.append(score)

plot_cgcs_vs_noise(
    noise_levels=over_levels,
    cgcs_values=cgcs_scores,
    title="CGCS vs Overexpression Level in Simulated Transcriptome"
)